# Services Integration Tests

Test all service modules end-to-end:
1. **LLM Service** — generation, review, structured output
2. **Config & Constants** — settings loading, env-configurable models

**Prerequisites:**
- `.env.develop` configured
- GCP auth configured (`gcloud auth application-default login`)

In [1]:
# Cell 1: Setup
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

Project root: /home/sakana/Code/ManifoldsTeam/AIServices


## 1. Config & Constants

In [2]:
# Cell 2: Verify settings & constants
from src.config import get_settings
from src.config.constants import (
    GENERATION_TEMPERATURE,
    GENERATION_MAX_TOKENS,
    REVIEW_TEMPERATURE,
    REVIEW_MAX_TOKENS,
    STRUCTURED_TEMPERATURE,
    STRUCTURED_MAX_TOKENS,
    LLM_MAX_RETRIES,
    ALLOWED_MIME_TYPES,
    MAX_FILE_SIZE_BYTES,
)

settings = get_settings()

print("=== Settings ===")
print(f"  ENV:              {settings.env}")
print(f"  GCP Project:      {settings.gcp_project_id}")
print(f"  GCP Location:     {settings.gcp_location}")
print(f"  Gen Model:        {settings.generation_model}")
print(f"  Review Model:     {settings.review_model}")
print(f"  Data Store ID:    {settings.data_store_id}")
print(f"  Search Engine ID: {settings.search_engine_id}")
print(f"  GCS Bucket:       {settings.gcs_bucket}")
print(f"  Log Level:        {settings.log_level}")

print("\n=== Constants ===")
print(f"  GENERATION_TEMPERATURE:  {GENERATION_TEMPERATURE}")
print(f"  GENERATION_MAX_TOKENS:   {GENERATION_MAX_TOKENS}")
print(f"  REVIEW_TEMPERATURE:      {REVIEW_TEMPERATURE}")
print(f"  REVIEW_MAX_TOKENS:       {REVIEW_MAX_TOKENS}")
print(f"  STRUCTURED_TEMPERATURE:  {STRUCTURED_TEMPERATURE}")
print(f"  STRUCTURED_MAX_TOKENS:   {STRUCTURED_MAX_TOKENS}")
print(f"  LLM_MAX_RETRIES:         {LLM_MAX_RETRIES}")
print(f"  MAX_FILE_SIZE_BYTES:      {MAX_FILE_SIZE_BYTES / 1024 / 1024} MB")
print(f"  ALLOWED_MIME_TYPES:      {ALLOWED_MIME_TYPES}")

=== Settings ===
  ENV:              develop
  GCP Project:      green-mercury-485016-n1
  GCP Location:     asia-southeast1
  Gen Model:        gemini-2.5-flash
  Review Model:     gemini-3.1-flash-lite-preview
  Data Store ID:    aiservice-datastore-m1_1772802306291
  Search Engine ID: gp-mathagent_1773042630372
  GCS Bucket:       documents-development-bucket
  Log Level:        DEBUG

=== Constants ===
  GENERATION_TEMPERATURE:  0.7
  GENERATION_MAX_TOKENS:   8192
  REVIEW_TEMPERATURE:      0.1
  REVIEW_MAX_TOKENS:       4096
  STRUCTURED_TEMPERATURE:  0.3
  STRUCTURED_MAX_TOKENS:   8192
  LLM_MAX_RETRIES:         3
  MAX_FILE_SIZE_BYTES:      50.0 MB
  ALLOWED_MIME_TYPES:      {'application/vnd.openxmlformats-officedocument.presentationml.presentation', 'application/pdf', 'application/vnd.openxmlformats-officedocument.wordprocessingml.document'}


In [3]:
# Cell 3: Verify computed properties
print("=== Computed Properties ===")
print(f"  data_store_path:      {settings.data_store_path}")
print(f"  search_engine_path:   {settings.search_engine_path}")
print(f"  cloud_tasks_queue_path: {settings.cloud_tasks_queue_path}")
print(f"  is_production:        {settings.is_production}")

=== Computed Properties ===
  data_store_path:      projects/green-mercury-485016-n1/locations/global/collections/default_collection/dataStores/aiservice-datastore-m1_1772802306291
  search_engine_path:   projects/green-mercury-485016-n1/locations/global/collections/default_collection/engines/gp-mathagent_1773042630372
  cloud_tasks_queue_path: projects/green-mercury-485016-n1/locations/asia-southeast1/queues/generation-queue
  is_production:        False


## 2. LLM Service — Generation

In [4]:
# Cell 4: Test get_generation_llm
from src.services.llm import get_generation_llm

gen_llm = get_generation_llm()
print(f"Generation LLM model: {gen_llm.model_name}")
print(f"Temperature: {gen_llm.temperature}")
print(f"Max output tokens: {gen_llm.max_output_tokens}")

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


2026-03-10 10:23:45 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.7
Generation LLM model: gemini-2.5-flash
Temperature: 0.7
Max output tokens: 8192


/home/sakana/Code/ManifoldsTeam/AIServices/src/services/llm.py:62: LangChainDeprecationWarning: The class `ChatVertexAI` was deprecated in LangChain 3.2.0 and will be removed in 4.0.0. An updated version of the class exists in the `langchain-google-genai package and should be used instead. To use it run `pip install -U `langchain-google-genai` and import as `from `langchain_google_genai import ChatGoogleGenerativeAI``.
  return ChatVertexAI(


In [5]:
# Cell 5: Invoke generation LLM with a simple prompt
response = gen_llm.invoke(
    "Tạo 1 câu hỏi trắc nghiệm toán lớp 10 về phương trình bậc hai. "
    "Chỉ trả lời ngắn gọn, 1 câu hỏi và 4 đáp án A B C D."
)

print(f"Response type: {type(response).__name__}")
print(f"Content length: {len(response.content)} chars")
print(f"\n--- Generated Content ---")
print(response.content)

if response.usage_metadata:
    um = response.usage_metadata
    print(f"\n--- Token Usage ---")
    if isinstance(um, dict):
        print(f"  Input tokens:  {um.get('input_tokens', '?')}")
        print(f"  Output tokens: {um.get('output_tokens', '?')}")
        print(f"  Total tokens:  {um.get('total_tokens', '?')}")
    else:
        print(f"  Input tokens:  {um.input_tokens}")
        print(f"  Output tokens: {um.output_tokens}")
        print(f"  Total tokens:  {um.total_tokens}")

Response type: AIMessage
Content length: 119 chars

--- Generated Content ---
**Câu hỏi:** Phương trình $x^2 - 5x + 7 = 0$ có bao nhiêu nghiệm thực?

A. 0 nghiệm
B. 1 nghiệm
C. 2 nghiệm
D. 3 nghiệm

--- Token Usage ---
  Input tokens:  40
  Output tokens: 51
  Total tokens:  1475


## 3. LLM Service — Review

In [6]:
# Cell 6: Test get_review_llm
from src.services.llm import get_review_llm

review_llm = get_review_llm()
print(f"Review LLM model: {review_llm.model_name}")
print(f"Location: {review_llm.location}")
print(f"Temperature: {review_llm.temperature}")
print(f"Max output tokens: {review_llm.max_output_tokens}")

2026-03-10 10:24:03 [debug    ] creating_review_llm            location=global max_output_tokens=4096 model=gemini-3.1-flash-lite-preview temperature=0.1
Review LLM model: gemini-3.1-flash-lite-preview
Location: global
Temperature: 0.1
Max output tokens: 4096


In [7]:
# Cell 7: Invoke review LLM for quality check
review_prompt = f"""Đánh giá chất lượng câu hỏi trắc nghiệm sau (1-10 điểm + nhận xét ngắn):

{response.content}

Trả lời ngắn gọn: Điểm/10 + 1 dòng nhận xét."""

review = review_llm.invoke(review_prompt)
print(f"--- Review ---")
print(review.content)
if review.usage_metadata:
    um = review.usage_metadata
    if isinstance(um, dict):
        print(f"\nTokens: in={um.get('input_tokens')}, out={um.get('output_tokens')}")
    else:
        print(f"\nTokens: in={um.input_tokens}, out={um.output_tokens}")

--- Review ---
[{'type': 'text', 'text': '**9/10**\n\nNhận xét: Câu hỏi rõ ràng, chính xác về mặt toán học, tuy nhiên phương án D (3 nghiệm) là một phương án gây nhiễu không cần thiết vì phương trình bậc hai không thể có 3 nghiệm.', 'thought_signature': 'AY89a19tIZhaDK5lEyMkzaDsicrDSzUH/I7efcXMIvvwd2fIQ8wSDX2x0G4ocgUq/kg='}]

Tokens: in=92, out=55


## 4. LLM Service — Structured Output

In [8]:
# Cell 8: Test get_structured_llm with Pydantic schema
from pydantic import BaseModel, Field
from src.services.llm import get_structured_llm


class QuizQuestion(BaseModel):
    """A multiple-choice quiz question."""
    question: str = Field(description="The question text")
    option_a: str = Field(description="Option A")
    option_b: str = Field(description="Option B")
    option_c: str = Field(description="Option C")
    option_d: str = Field(description="Option D")
    correct_answer: str = Field(description="The correct option letter (A/B/C/D)")
    explanation: str = Field(description="Brief explanation of the answer")


structured_llm = get_structured_llm(output_schema=QuizQuestion)
print(f"Structured LLM ready (schema: QuizQuestion)")

2026-03-10 10:24:20 [debug    ] creating_generation_llm        location=asia-southeast1 max_output_tokens=8192 model=gemini-2.5-flash temperature=0.3
Structured LLM ready (schema: QuizQuestion)


In [9]:
# Cell 9: Invoke structured LLM
result = structured_llm.invoke(
    "Tạo 1 câu hỏi trắc nghiệm hóa học lớp 10 về cấu tạo nguyên tử."
)

print(f"Result type: {type(result).__name__}")
print(f"\nQuestion: {result.question}")
print(f"A: {result.option_a}")
print(f"B: {result.option_b}")
print(f"C: {result.option_c}")
print(f"D: {result.option_d}")
print(f"Correct: {result.correct_answer}")
print(f"Explanation: {result.explanation}")

Result type: QuizQuestion

Question: Hạt nào sau đây được tìm thấy trong hạt nhân của nguyên tử?
A: Proton và electron
B: Neutron và electron
C: Proton và neutron
D: Proton, neutron và electron
Correct: C
Explanation: Hạt nhân nguyên tử bao gồm các hạt proton mang điện tích dương và các hạt neutron không mang điện. Các hạt electron mang điện tích âm chuyển động xung quanh hạt nhân.


## 5. End-to-end: Retrieval + LLM Generation

In [10]:
# Cell 10: Full pipeline — retrieve context then generate question
from src.services.vertex_search import retrieve_context

# Step 1: Retrieve relevant context
context_strings, source_metadata = await retrieve_context(
    user_id="__system__",
    query="phương trình bậc hai toán lớp 10",
    doc_scope="system",
    max_documents=3,
)
context_text = "\n---\n".join(context_strings[:3])
print(f"Retrieved {len(context_strings)} context chunks ({len(context_text)} chars)")
print(f"Sources: {[m.get('source', '?')[-40:] for m in source_metadata]}")

2026-03-10 10:24:33 [info     ] retrieving_context             doc_scope=system query='phương trình bậc hai toán lớp 10' user_id=__system__
2026-03-10 10:24:33 [debug    ] creating_retriever             doc_scope=system extractive_answers=True filter='user_id: ANY("__system__")' search_engine_id=gp-mathagent_1773042630372 user_id=__system__
2026-03-10 10:24:33 [debug    ] retrieving_documents           doc_scope=system query='phương trình bậc hai toán lớp 10'


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-03-10 10:24:35 [debug    ] retrieved_documents            count=6
2026-03-10 10:24:35 [info     ] context_retrieved              chunks=6 sources=6
Retrieved 6 context chunks (557 chars)
Sources: ['ser/2026-03-07/test/lesson_plan_user.txt', '/tin-hoc/sach-giao-khoa-tin-hoc-10.pdf50', '0/tin-hoc/sach-giao-khoa-tin-hoc-10.pdf7', 'sach-giao-khoa-vat-li-10-nang-cao.pdf130', '/sach-giao-khoa-vat-li-10-nang-cao.pdf36', '/sach-giao-khoa-vat-li-10-nang-cao.pdf18']


In [11]:
# Cell 11: Generate structured quiz question from retrieved context
prompt = f"""Dựa trên nội dung sách giáo khoa dưới đây, tạo 1 câu hỏi trắc nghiệm
toán lớp 10 về phương trình bậc hai.

Nội dung tham khảo:
{context_text[:2000]}

Yêu cầu: Câu hỏi phải liên quan đến nội dung đã cho."""

quiz_result = structured_llm.invoke(prompt)

print("=== Generated Quiz (RAG) ===")
print(f"Q: {quiz_result.question}")
print(f"A: {quiz_result.option_a}")
print(f"B: {quiz_result.option_b}")
print(f"C: {quiz_result.option_c}")
print(f"D: {quiz_result.option_d}")
print(f"Answer: {quiz_result.correct_answer}")
print(f"Why: {quiz_result.explanation}")

=== Generated Quiz (RAG) ===
Q: Phương trình bậc hai tổng quát có dạng như thế nào?
A: ax + b = 0
B: ax² + bx + c = 0 (a ≠ 0)
C: ax³ + bx² + cx + d = 0
D: ax² + bx + c = 0 (a = 0)
Answer: B
Why: Theo nội dung sách giáo khoa, phương trình bậc hai tổng quát có dạng ax² + bx + c = 0 với a ≠ 0.
